Run following command in terminal:

sbatch --gpus=2 --gres=gpumem:40g --time=05:00:00 --mem-per-cpu=32g --wrap="jupyter nbconvert --to notebook --execute 01_251029_generating_description_Apertus-8B-Instruct-2509s.ipynb --inplace"

In [1]:
import pandas as pd

initial_groups_df = pd.read_excel('../data/251027 input data points and groups.xlsx')
initial_groups_df['Data_groups'] = initial_groups_df['Data_groups']\
    .str.strip().str.lower().str.replace('&', 'and')
initial_groups_df.head()

,Data_groups,Data_points,Source,File,Passport_type
0,general information,"Product commercial name, Manufacturer's name, ...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
1,material health,"Security information, warnings, recommendation...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
2,sustainability,"Environmental declaration, Life cycle assessme...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
3,design and production,"Manufacturing process, Manufacturing technique...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
4,use and operate phase,"Positioning in the building, location in the b...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport


In [2]:
initial_points_df = initial_groups_df.assign(Data_points=initial_groups_df['Data_points']\
    .str.replace('&', 'and').str.split(',')).explode('Data_points').reset_index()
initial_points_df['Data_points'] = initial_points_df['Data_points']\
    .str.strip().str.lower()

initial_points_df = initial_points_df[initial_points_df["Data_points"] != ""].reset_index(drop=True)

initial_points_df.head()

,index,Data_groups,Data_points,Source,File,Passport_type
0,0,general information,product commercial name,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
1,0,general information,manufacturer's name,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
2,0,general information,manufacturer's details,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
3,0,general information,materials composition,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
4,0,general information,product properties,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport


In [3]:
initial_points_df.shape

(2010, 6)

In [4]:
import os

count = 0

files = [f for f in os.listdir("../data/literature/") if os.path.isfile(os.path.join("../data/literature/", f))]

for file in files:
    if file not in initial_points_df.File.unique():
        print (file)
len(files)

30

In [6]:
for source in initial_points_df.File.unique():
    if os.path.isfile(f"../data/literature/{source}"):
        count += 1
    else:
        print("Missing source file!")
count, initial_points_df.File.unique().shape

(30, (30,))

In [7]:
initial_points_df['description'] = ""
initial_points_df.tail()

,index,Data_groups,Data_points,Source,File,Passport_type,description
2005,324,essential environmental characteristics,eco-toxicity,CPR 2024,CPR 2024.pdf,Digital product passport,
2006,324,essential environmental characteristics,freshwater,CPR 2024,CPR 2024.pdf,Digital product passport,
2007,324,essential environmental characteristics,human toxicity cancerogenic,CPR 2024,CPR 2024.pdf,Digital product passport,
2008,324,essential environmental characteristics,human toxicity non-cancerogenic,CPR 2024,CPR 2024.pdf,Digital product passport,
2009,324,essential environmental characteristics,land use related impacts,CPR 2024,CPR 2024.pdf,Digital product passport,


In [8]:
from collections import Counter
Counter(initial_points_df.Passport_type)

Counter({'Material passport': 849,
         'Digital product passport': 818,
         'Digital passport': 185,
         'Product Circularity Data Sheet': 158})

In [9]:
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [12]:
from llama_index.core import (
    Settings, 
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
import os

# --- CONFIG ---
EMBED_MODEL = "BAAI/bge-m3"  # embedding model on Hugging Face
LLM_MODEL = "gpt-5-nano-2025-08-07"  # Apertus Instruct 8B on HF

In [13]:
# --- Embeddings (Hugging Face) ---
# HuggingFaceEmbedding will load local model if available, otherwise use HF hub.
embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL)

# TODO: load the embedding model in llama-index setting
Settings.embed_model = embed_model

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-m3
Load pretrained SentenceTransformer: BAAI/bge-m3


In [ ]:
import os

# Provide your OpenAI API key via the environment before running this notebook, e.g.
#   export OPENAI_API_KEY="sk-..."   (Linux/macOS)
#   $env:OPENAI_API_KEY = "sk-..."   (PowerShell)
assert os.environ.get("OPENAI_API_KEY"), "Set the OPENAI_API_KEY environment variable"

In [19]:
from llama_index.core import PromptTemplate
from llama_index.llms.openai import OpenAI

# set the LLM
system_prompt = """
You are a helpful assistant with expertise in sustainable construction.

Rules:
1. Use exactly three sentences. 
2. Base your answer on the provided text.
2. Do not add extra explanations, commentary, or unrelated information.
"""
query_wrapper_prompt = PromptTemplate("Instruction: {query_str}\nResponse:")

# --- OpenAI LLM ---
openai_llm = OpenAI(
    model="gpt-5-nano-2025-08-07", 
    temperature=0.1,
    max_tokens=256,
    system_prompt=system_prompt,
    query_wrapper_prompt=query_wrapper_prompt
)


# Settings.llm = hf_llm = HuggingFaceLLM(
#     model_name="swiss-ai/Apertus-8B-Instruct-2509",
#     tokenizer_name="swiss-ai/Apertus-8B-Instruct-2509",
#     system_prompt=system_prompt,
#     query_wrapper_prompt=query_wrapper_prompt,
#     generate_kwargs={"temperature": 0.1,
#                      "do_sample": True, # sampling required for temperature to work
#                      "top_p": 0.9},
#     device_map="auto",
#     max_new_tokens=256,
#     context_window=4096,
# )

In [15]:
import os
import logging
from pypdf import PdfReader

pdf_dir = "../data/literature"   # your PDF directory
warning_files = []

# --- Custom log handler ---
class WarningCaptureHandler(logging.Handler):
    def __init__(self):
        super().__init__()
        self.captured = False

    def emit(self, record):
        msg = record.getMessage()
        if "Ignoring wrong pointing object" in msg:
            self.captured = True

# --- Attach handler to pypdf logger ---
logger = logging.getLogger("pypdf._reader")
logger.setLevel(logging.WARNING)

for filename in os.listdir(pdf_dir):
    if not filename.lower().endswith(".pdf"):
        continue

    filepath = os.path.join(pdf_dir, filename)
    handler = WarningCaptureHandler()
    logger.addHandler(handler)

    try:
        reader = PdfReader(filepath)
        _ = len(reader.pages)
    except Exception as e:
        # still catch real parse errors
        print(f"[ERROR] {filename} failed to parse: {e}")
    finally:
        if handler.captured:
            warning_files.append(filename)
        logger.removeHandler(handler)

# --- Report PDFs that caused warnings ---
print("\nPDFs triggering 'Ignoring wrong pointing object' warning:")
for f in warning_files:
    print(f" - {f}")




PDFs triggering 'Ignoring wrong pointing object' warning:


In [17]:
from pathlib import Path
from llama_index.readers.file import PyMuPDFReader

file_dir = "../data/literature" 
persist_dir = "../storage/lyft"

# For testing only
# Settings.llm = None


# Directory containing PDFs
pdf_dir = Path(file_dir)

# Instantiate PyMuPDFReader
pdf_reader = {".pdf": PyMuPDFReader()}

# Pass it to SimpleDirectoryReader
reader = SimpleDirectoryReader(
    input_dir=pdf_dir,   # directory or list of files
    file_extractor=pdf_reader
)

# Load all documents
documents = reader.load_data()

print(f"Loaded {len(documents)} documents.")
print(documents[0].text[:500])


Loaded 1012 documents.
Journal of Building Engineering 43 (2021) 103233
Available online 3 September 2021
2352-7102/© 2021 Elsevier Ltd. All rights reserved.
Digitizing material passport for sustainable construction projects using BIM 
Islam Atta a, Emad S. Bakhoum b,c,*, Mohamed M. Marzouk d 
a Teaching Assistant, Civil Engineering Department, Al-Madina Higher Institute for Engineering and Technology, Giza, Egypt 
b Civil Infrastructure Engineering and Management Department, Nile University, Giza, Egypt 
c Civil Engi


In [ ]:
index = VectorStoreIndex.from_documents(
    documents,
)

index.storage_context.persist(persist_dir="../storage/lyft")

In [22]:
query_engine = index.as_query_engine(response_mode="compact") 

In [24]:
from pprint import pprint
import time
import json
import os

descriptions = []
descriptions_per_point = {}
responses = {}

all_references = []
response_dir = "./251030_generated_descriptions_GPT5-nano"
for i in range(initial_points_df.shape[0]):
    if os.path.exists(f"{response_dir}/{i}_response.json"):
        continue
    file = initial_points_df.iloc[i].File
        
    data_group = initial_points_df.iloc[i].Data_groups
    data_point = initial_points_df.iloc[i].Data_points

    prompt = f"What is the meaning of {data_point} in relation to {data_group}?"
    
    response = query_engine.query(prompt)

    print(f'index: {i}')
    
    point_i_response = {}

    point_i_response["question"] =  prompt
    point_i_response["original_source"] =  file
    point_i_response["data_group"] =  data_group
    point_i_response["data_point"] =  data_point


    references = []
    for node_with_score in response.source_nodes:
        node = node_with_score.node  # Access the underlying Node
        reference = {}
        reference["text"] = node.get_text() # Node text 
        reference["metadata"] = node.metadata  # Metadata if any
        reference["score"] = node_with_score.score # Similarity score
        references.append(reference)

    point_i_response["reference_1"] =  references[0]["metadata"]["file_name"]
    point_i_response["reference_2"] =  references[1]["metadata"]["file_name"]
    point_i_response["description"] =  response.response
    point_i_response["references"] = references

    with open(f"{response_dir}/{i}_response.json", "w") as f:
        json.dump(point_i_response, f)
    
    descriptions.append(response.response)
    descriptions_per_point[data_point] = response.response
    responses[i] = response
    initial_points_df.loc[i, 'description'] = response.response
    
    all_references.append(point_i_response["reference_1"])
    all_references.append(point_i_response["reference_2"])



    


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
index: 1
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
index: 2
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
index: 3
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
index: 4
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
index: 5
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Requ

In [ ]:
from pprint import pprint
import time
import json
import os

descriptions = []
descriptions_per_point = {}
responses = {}

all_references = []

for i in range(initial_points_df.shape[0]):
    if os.path.exists(f"./generated_descriptions_Apertus-8B-Instruct-2509s/{i}_response.json"):
        continue
    file = initial_points_df.iloc[i].File
        
    data_group = initial_points_df.iloc[i].Data_groups
    data_point = initial_points_df.iloc[i].Data_points

    prompt = f"What is the meaning of {data_point} in relation to {data_group}?"
    
    # query_engine = documents[file].as_query_engine()
    response = query_engine.query(prompt)

    print(f'index: {i}')
    
    point_i_response = {}

    point_i_response["question"] =  prompt
    point_i_response["original_source"] =  file
    point_i_response["data_group"] =  data_group
    point_i_response["data_point"] =  data_point


    references = []
    for node_with_score in response.source_nodes:
        node = node_with_score.node  # Access the underlying Node
        reference = {}
        reference["text"] = node.get_text() # Node text 
        reference["metadata"] = node.metadata  # Metadata if any
        reference["score"] = node_with_score.score # Similarity score
        references.append(reference)

    point_i_response["reference_1"] =  references[0]["metadata"]["file_name"]
    point_i_response["reference_2"] =  references[1]["metadata"]["file_name"]
    point_i_response["description"] =  response.response
    point_i_response["references"] = references

    with open(f"./generated_descriptions_Apertus-8B-Instruct-2509s/{i}_response.json", "w") as f:
        json.dump(point_i_response, f)
    
    descriptions.append(response.response)
    descriptions_per_point[data_point] = response.response
    responses[i] = response
    initial_points_df.loc[i, 'description'] = response.response
    
    all_references.append(point_i_response["reference_1"])
    all_references.append(point_i_response["reference_2"])
    
    break
    


index: 0


In [13]:
Counter(all_references)

Counter({'CPR 2024.pdf': 2})

In [14]:
len(Counter(all_references).keys())

for file in files:
    if file not in Counter(all_references).keys():
        print (file)

Munaro and Tavares 2021.pdf
Stratmann 2023.pdf
Göswein 2022.pdf
Atta 2021.pdf
Bauen digital Schweiz 2024.pdf
Bosma 2024.pdf
Circularise 2025.pdf
Byers 2025.pdf
Çetin 2023.pdf
Giovanardi 2023.pdf
Heisel and Rau-Oberhuber 2020.pdf
Jensen 2023.pdf
Honic 2021.pdf
ISO 59040 2025.pdf
KC 2024.pdf
Honic 2019.pdf
Lopes and Barata 2024.pdf
Kebede 2024.pdf
Seddiqui 2024.pdf
Mulhall 2022.pdf
Mao and Cao 2025.pdf
Markou 2025.pdf
Platform CB 2023.pdf
Wan and Jiang 2025.pdf
Van Capelleveen 2023.pdf
Christensen 2025.pdf
BAMB 2019.pdf
Ruismäki 2025.pdf
ESPR 2024.pdf


In [ ]:
# Save to Excel
initial_points_df.to_excel("251030 descriptions GPT5-nano of input data points and groups.xlsx", index=False)